# Stage 04: Data Acquisition and Ingestion

**Project:** SPY Next-Day High-Volatility Risk Alert  
**Date:** 2026-08-23

This notebook completes the Stage 04 starter tasks: acquire market data through an API, scrape a permitted public table, parse both results into typed pandas DataFrames, validate them, and preserve timestamped raw snapshots in `data/raw/`.

In [1]:
# Packages needed (run once in the Stage 02 environment if missing):
# %pip install pandas requests beautifulsoup4 python-dotenv

## 1. Reproducible paths and safe configuration

The notebook searches upward for `homework/homework04`, so it can be executed from the repository root or from its own folder. Real secrets belong only in the ignored `.env`; `.env.example` documents variable names without values. The selected sources do not require an API key.

In [2]:
from __future__ import annotations

import os
import sys
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv


def locate_homework_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        if candidate.name == "homework04" and (candidate / "src").exists():
            return candidate
        nested = candidate / "homework" / "homework04"
        if (nested / "src").exists():
            return nested
    raise FileNotFoundError("Could not locate homework/homework04")


ROOT = locate_homework_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(dotenv_path=ROOT / ".env")
raw_setting = Path(os.getenv("DATA_DIR_RAW", "data/raw")).expanduser()
RAW = raw_setting if raw_setting.is_absolute() else ROOT / raw_setting
RAW.mkdir(parents=True, exist_ok=True)

repo_gitignore = ROOT.parents[1] / ".gitignore"
env_is_ignored_by_rule = repo_gitignore.exists() and ".env" in repo_gitignore.read_text()

print("Homework root:", ROOT)
print("Raw data directory:", RAW)
print("Local .env present:", (ROOT / ".env").exists())
print(".env covered by repository ignore rule:", env_is_ignored_by_rule)
print("Alpha Vantage key loaded (value never displayed):", bool(os.getenv("ALPHAVANTAGE_API_KEY")))

Homework root: /Users/cengchengyu/Documents/NYU/Boot Camp/CS HW/Project/homework/homework04
Raw data directory: /Users/cengchengyu/Documents/NYU/Boot Camp/CS HW/Project/homework/homework04/data/raw
Local .env present: True
.env covered by repository ignore rule: True
Alpha Vantage key loaded (value never displayed): False


In [3]:
from src.ingestion import (
    SP500_CONSTITUENTS_URL,
    build_manifest,
    fetch_nasdaq_history,
    scrape_sp500_constituents,
    timestamp_utc,
    validate_sp500_constituents,
    validate_spy_history,
    write_raw_csv,
)

RUN_TIMESTAMP = timestamp_utc()
RUN_TIMESTAMP

'20260823-1503'

## 2. Part 1 — API pull (required)

**Source:** Nasdaq public historical-data JSON endpoint  
**Endpoint:** `https://api.nasdaq.com/api/quote/SPY/historical`  
**Parameters:** ETF asset class, SPY, a ten-year date window, and a 5,000-row limit.  
**Purpose:** daily OHLCV observations are the core input for the project's future return, volatility, and liquidity features.

The response is rejected if the HTTP request fails, Nasdaq reports a non-200 API status, or the row collection is empty. Numeric fields are parsed strictly after treating source `N/A` values as missing. Isolated incomplete rows are counted in metadata and excluded; the run fails if they exceed 1% of the response.

In [4]:
SYMBOL = "SPY"
START_DATE = "2016-08-23"
END_DATE = datetime.now(UTC).date().isoformat()

df_api, api_metadata = fetch_nasdaq_history(
    SYMBOL,
    START_DATE,
    END_DATE,
)
print("Request metadata:", api_metadata)
print("Shape:", df_api.shape)
display(df_api.head(3))
display(df_api.tail(3))
display(df_api.dtypes.rename("dtype").to_frame())

Request metadata: {'source': 'Nasdaq', 'source_url': 'https://api.nasdaq.com/api/quote/SPY/historical', 'request_params': {'assetclass': 'etf', 'fromdate': '2016-08-23', 'todate': '2026-08-23', 'limit': '5000'}, 'reported_total_records': 2513, 'dropped_incomplete_rows': 1, 'dropped_incomplete_dates': ['2026-04-20'], 'retrieved_at_utc': '2026-08-23T15:03:02.815481+00:00'}
Shape: (2512, 6)


,date,open,high,low,close,volume
0,2016-08-23,219.25,219.6,218.9,218.97,53289030
1,2016-08-24,218.8,218.91,217.36,217.85,71553010
2,2016-08-25,217.4,218.19,217.22,217.7,69128600


,date,open,high,low,close,volume
2509,2026-08-19,770.36,772.47,768.1,769.06,40306020
2510,2026-08-20,765.96,768.15,762.04,762.6,45520300
2511,2026-08-21,766.05,767.85,764.17,765.72,39188670


,dtype
date,datetime64[us]
open,Float64
high,Float64
low,Float64
close,Float64
volume,Int64


In [5]:
api_validation = validate_spy_history(df_api)
display(pd.Series(api_validation, name="value").to_frame())

,value
shape,"[2512, 6]"
required_columns_present,True
dtypes,"{'date': 'datetime64[us]', 'open': 'Float64', ..."
na_by_column,"{'date': 0, 'open': 0, 'high': 0, 'low': 0, 'c..."
duplicate_dates,0
date_min,2016-08-23
date_max,2026-08-21
invalid_ohlc_rows,0
passed,True


In [6]:
api_path = write_raw_csv(
    df_api,
    RAW,
    f"api_NASDAQ_{SYMBOL}",
    timestamp=RUN_TIMESTAMP,
)
print("Saved API snapshot:", api_path.relative_to(ROOT))

Saved API snapshot: data/raw/api_NASDAQ_SPY_20260823-1503.csv


## 3. Part 2 — scrape a public table (required)

**Source:** Wikipedia, *List of S&P 500 companies*  
**URL:** <https://en.wikipedia.org/wiki/List_of_S%26P_500_companies>  
**Selector:** `table#constituents`  
**Purpose:** the current constituents and GICS classifications provide auditable market-context data that can support later sector-level diagnostics.

The page is public and exposes a conventional HTML table. The request identifies this educational use through a descriptive user agent. No login, paywall, or access-control mechanism is bypassed.

In [7]:
df_scrape, scrape_metadata = scrape_sp500_constituents()
print("Request metadata:", scrape_metadata)
print("Shape:", df_scrape.shape)
display(df_scrape.head(3))
display(df_scrape.dtypes.rename("dtype").to_frame())

Request metadata: {'source': 'Wikipedia', 'source_url': 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies', 'table_selector': 'table#constituents', 'retrieved_at_utc': '2026-08-23T15:03:03.782472+00:00'}
Shape: (503, 8)


,symbol,security,gics_sector,gics_sub_industry,headquarters_location,date_added,cik,founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee , Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888


,dtype
symbol,str
security,str
gics_sector,str
gics_sub_industry,str
headquarters_location,str
date_added,datetime64[us]
cik,int64
founded,str


In [8]:
scrape_validation = validate_sp500_constituents(df_scrape)
display(pd.Series(scrape_validation, name="value").to_frame())

,value
shape,"[503, 8]"
required_columns_present,True
dtypes,"{'symbol': 'str', 'security': 'str', 'gics_sec..."
na_by_column,"{'symbol': 0, 'security': 0, 'gics_sector': 0,..."
blank_text,"{'symbol': 0, 'security': 0, 'gics_sector': 0,..."
duplicate_symbols,0
unique_sectors,11
numeric_cik,True
passed,True


In [9]:
scrape_path = write_raw_csv(
    df_scrape,
    RAW,
    "scrape_WIKIPEDIA_SP500_CONSTITUENTS",
    timestamp=RUN_TIMESTAMP,
)
print("Saved scrape snapshot:", scrape_path.relative_to(ROOT))

Saved scrape snapshot: data/raw/scrape_WIKIPEDIA_SP500_CONSTITUENTS_20260823-1503.csv


## 4. Manifest and reproducibility record

The manifest binds each raw file to its source metadata, validation result, byte size, and SHA-256 checksum. This supplements—not replaces—the unchanged raw CSV snapshots.

In [10]:
manifest_path = build_manifest(
    [
        {
            "path": api_path,
            "dataset": "SPY daily OHLCV",
            "rows": len(df_api),
            "columns": list(df_api.columns),
            "source_metadata": api_metadata,
            "validation": api_validation,
        },
        {
            "path": scrape_path,
            "dataset": "Current S&P 500 constituents",
            "rows": len(df_scrape),
            "columns": list(df_scrape.columns),
            "source_metadata": scrape_metadata,
            "validation": scrape_validation,
        },
    ],
    RAW / f"ingestion_manifest_{RUN_TIMESTAMP}.json",
)
print("Saved manifest:", manifest_path.relative_to(ROOT))

Saved manifest: data/raw/ingestion_manifest_20260823-1503.json


## 5. Assumptions and risks

- **Availability and rate limits:** Nasdaq's public website endpoint is not a guaranteed service. Access limits, response schemas, and availability can change. A future failure should stop the pipeline rather than silently use an invented fallback.
- **Unadjusted prices:** Nasdaq supplies unadjusted OHLCV in this extract. Later return work must explicitly choose and document a corporate-action adjustment policy.
- **Observed incomplete row:** in the submitted run, Nasdaq reported `N/A` volume for 2026-04-20. The parser recorded that date and excluded the single incomplete row; it was not imputed or silently converted to zero.
- **Selector fragility:** the Wikipedia parser relies on `table#constituents` and named columns. Page redesigns or renamed fields will cause an intentional validation error.
- **Snapshot semantics:** the constituent list reflects the current page at retrieval time, not historical membership. Using it as a historical universe would create survivorship bias.
- **Revisions:** either source may revise records later. Timestamped CSVs and checksums preserve the submitted inputs, but rerunning on a later date is expected to produce a different snapshot.
- **Scope:** these data support predictive risk analysis and descriptive diagnostics. They do not identify causes of volatility and do not constitute investment advice.

The local `.env` is present for safe configuration and covered by the repository's `.env` ignore rule. No secret value is printed or committed.

In [11]:
summary = pd.DataFrame(
    [
        {
            "dataset": "Nasdaq SPY OHLCV",
            "rows": len(df_api),
            "columns": len(df_api.columns),
            "validation_passed": api_validation["passed"],
            "raw_file": api_path.name,
        },
        {
            "dataset": "Wikipedia S&P 500 constituents",
            "rows": len(df_scrape),
            "columns": len(df_scrape.columns),
            "validation_passed": scrape_validation["passed"],
            "raw_file": scrape_path.name,
        },
    ]
)
display(summary)
assert summary["validation_passed"].all()
assert api_path.exists() and scrape_path.exists() and manifest_path.exists()
print("Stage 04 acquisition, validation, and raw-file checks passed.")

,dataset,rows,columns,validation_passed,raw_file
0,Nasdaq SPY OHLCV,2512,6,True,api_NASDAQ_SPY_20260823-1503.csv
1,Wikipedia S&P 500 constituents,503,8,True,scrape_WIKIPEDIA_SP500_CONSTITUENTS_20260823-1...


Stage 04 acquisition, validation, and raw-file checks passed.
